# Module 4 — Physics-Informed Neural Networks

**NHERI SimCenter/DesignSafe 2026 SPARC · Day 3, Session 3b**

**Exercise** [![Try on DesignSafe](https://raw.githubusercontent.com/DesignSafe-CI/training-ai/main/DesignSafe-Badge.svg)](https://jupyter.designsafe-ci.org/hub/user-redirect/lab/tree/CommunityData/Training/2026-SPARC/Day3/Session3b/04-pinn-cantilever-exercise.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DesignSafe-CI/training-ai/blob/main/04-pinn/04-pinn-cantilever-exercise.ipynb)

**Solution** [![Try on DesignSafe](https://raw.githubusercontent.com/DesignSafe-CI/training-ai/main/DesignSafe-Badge.svg)](https://jupyter.designsafe-ci.org/hub/user-redirect/lab/tree/CommunityData/Training/2026-SPARC/Day3/Session3b/04-pinn-cantilever.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DesignSafe-CI/training-ai/blob/main/04-pinn/04-pinn-cantilever.ipynb)

---

## Fitting sparse data alone gets the boundary wrong

Module 2 ended with a network fitted to eight deflection readings. It passed
through all eight and was still wrong: the slope at the clamped support came out
nonzero, and past the tip it wandered off. Adding two boundary terms to the loss
fixed the support.

We also know the governing equation for that beam, and we ignored it.

This module puts the whole equation in the loss.

## The network stops being a surrogate and becomes the solution

Everything in Modules 2 and 3 was a **surrogate**: fit a flexible function to
input-output pairs, then interrogate it. A PINN is a different object.

| | Surrogate (Modules 2-3) | PINN (this module) |
| --- | --- | --- |
| Network input | design parameters $(M, L, E)$ | a **coordinate** $x$ |
| Network output | a quantity of interest | the **field** $w(x)$ |
| Trained on | labelled simulation runs | the **PDE residual** |
| Labelled data needed | thousands | **zero** (optional) |
| The network is | an approximation of a solver | the *solution itself* |

That last row is the one to sit with. $w_\theta(x)$ is not a model *of* the
answer — it is a mesh-free ansatz *for* the answer, and training is the act of
solving the differential equation.

<img src="figs/pinn.png" width="640"/>

## Five parts: residual, soft BCs, hard BCs, inverse, limits

| Part | |
| --- | --- |
| 1 | Build the residual with autograd, and check it |
| 2 | Solve the beam with **no data at all** (soft BCs) |
| 3 | Hard constraints: satisfy the BCs *exactly*, by construction |
| 4 | **Inverse problem** — recover $EI$ from 8 noisy readings |
| 5 | When PINNs are the wrong tool |

## Setup

In [ ]:
%pip install torch matplotlib numpy --quiet

In [ ]:
%matplotlib inline

import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

RNG = 42
torch.manual_seed(RNG)
np.random.seed(RNG)
torch.set_default_dtype(torch.float64)   # 2nd derivatives deserve float64

plt.rcParams.update({"figure.figsize": (11, 4), "font.size": 11})
print("torch", torch.__version__)

## A tip-loaded cantilever, in moment-curvature form

Same cantilever as Module 2: tip-loaded, clamped at $x = 0$.

$$EI\,\frac{d^2w}{dx^2} = M(x) = -P(L - x), \qquad w(0) = 0, \quad w'(0) = 0$$

We use the moment-curvature (second-order) form rather than
$EI\,w'''' = q$. Both describe the same beam, but the second-order form needs
only two derivatives of the network instead of four — and every extra
differentiation amplifies noise in the network's output. We come back to the
fourth-order version in Part 5.

The exact solution, for scoring ourselves:

$$w(x) = -\frac{Px^2}{6EI}(3L - x)$$

### First, non-dimensionalise — this is not optional

With SI numbers, $EI = 2\times10^{6}$ and the deflection is about $0.02$. The
residual $EI\,w'' + P(L-x)$ has magnitude $\sim\!5000$ while the quantity we want
is $\sim\!0.02$ — five orders of magnitude apart. Gradient descent on that is
hopeless, and this is the single most common reason a hand-rolled PINN refuses to
train.

Scale the problem instead. With

$$\xi = \frac{x}{L}, \qquad W = \frac{w}{PL^3/3EI}$$

the equation becomes clean and $O(1)$:

$$\boxed{W''(\xi) = -3(1 - \xi)}, \qquad W(0) = W'(0) = 0,
\qquad W_{\text{exact}}(\xi) = -\tfrac{1}{2}\left(3\xi^2 - \xi^3\right)$$

Everything below solves that. We convert back to millimetres only for plotting.

In [ ]:
P, L, E, I = 1000.0, 5.0, 200e9, 1e-5      # N, m, Pa, m^4
EI = E * I
W_SCALE = P * L**3 / (3 * EI)               # tip deflection magnitude, m

print(f"EI       = {EI:.3e} N m^2")
print(f"W_SCALE  = {W_SCALE * 1000:.4f} mm   (tip deflection)")


def W_exact(xi):
    """Non-dimensional exact solution."""
    return -0.5 * (3 * xi**2 - xi**3)


def Wpp_exact(xi):
    """Its second derivative, = -3(1 - xi)."""
    return -3.0 * (1.0 - xi)


xi_plot = np.linspace(0, 1, 300)
print(f"\ncheck W_exact(1) = {W_exact(1.0):.4f}   (should be -1 by construction)")
print(f"physical tip     = {W_exact(1.0) * W_SCALE * 1000:.4f} mm")

## Part 1 — The residual, via automatic differentiation

In Module 2 we differentiated a network with respect to its input to get
$\partial T/\partial L$. The same call, applied twice, gives us $W''$.

The one detail that matters: `create_graph=True`. Without it the first derivative
is a leaf with no history, and the second `grad` call fails. With it, the
derivative is itself part of the graph — which is also what lets us
backpropagate *through* the residual to the weights.

In [ ]:
def deriv(y, x, n=1):
    """TODO: return the n-th derivative of y w.r.t. x.
    Call torch.autograd.grad n times. You MUST pass create_graph=True,
    or the second call will fail -- think about why.
    """
    ...


def make_net(width=32, depth=4):
    """TODO: a tanh MLP mapping 1 input -> 1 output.
    Why tanh and not ReLU, given that we need a second derivative?
    """
    ...


# Check your deriv() against the analytic W'' = -3(1 - xi).
xt = torch.linspace(0, 1, 5, requires_grad=True).unsqueeze(1)
y = -0.5 * (3 * xt**2 - xt**3)

Exact to machine precision — AD is not finite differences, there is no
step size and no truncation error.

### The residual needs no measured value of W

The residual of $W'' + 3(1-\xi) = 0$, evaluated at **collocation points**
scattered through the domain. Note what is absent: any measured value of $W$.

In [ ]:
def pde_residual(model, xi):
    """TODO: return W'' + 3(1 - xi) for the network `model`.
    Remember to set requires_grad on xi.
    """
    ...


def physics_loss(model, xi_colloc):
    """TODO: mean squared residual."""
    ...

## Part 2 — Solving the beam with no data

The full loss has two parts and no data term at all:

$$\mathcal{L} = \underbrace{\frac{1}{N_c}\sum_i r(\xi_i)^2}_{\text{PDE residual}}
\;+\;
\lambda_{BC}\underbrace{\left[W(0)^2 + W'(0)^2\right]}_{\text{boundary conditions}}$$

This is the "soft" or penalty approach: the boundary conditions are *encouraged*,
not enforced. $\lambda_{BC}$ decides how much they matter relative to the
equation, and choosing it is a genuine nuisance — we quantify that shortly.

In [ ]:
def bc_loss(model):
    """TODO: W(0)^2 + W'(0)^2 for the clamped end."""
    ...


def train_soft(model, n_colloc=100, epochs=5000, lr=5e-3, lam_bc=100.0,
               resample=False, quiet=False):
    """TODO: Adam loop minimising  physics_loss + lam_bc * bc_loss.
    Record the two loss components separately -- you will want to see
    which one is actually being minimised.
    """
    ...


torch.manual_seed(RNG)
soft = make_net()
h_soft = train_soft(soft, lam_bc=100.0, epochs=5000)

In [ ]:
def predict(model, xi_np):
    with torch.no_grad():
        return model(torch.tensor(xi_np).unsqueeze(1)).numpy().ravel()


W_soft = predict(soft, xi_plot)
err_soft = np.abs(W_soft - W_exact(xi_plot))

fig, ax = plt.subplots(1, 3, figsize=(14.5, 4))

ax[0].plot(xi_plot, W_exact(xi_plot) * W_SCALE * 1000, "k--", lw=2.5, label="exact")
ax[0].plot(xi_plot, W_soft * W_SCALE * 1000, "crimson", lw=2, label="PINN (no data)")
ax[0].set(xlabel=r"$\xi = x/L$", ylabel="deflection (mm)",
          title="Solved from the equation alone")
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].semilogy(h_soft["pde"], label="PDE residual")
ax[1].semilogy(h_soft["bc"], label="BC")
ax[1].set(xlabel="epoch", ylabel="loss", title="Loss components")
ax[1].legend(); ax[1].grid(alpha=.3)

ax[2].semilogy(xi_plot, np.maximum(err_soft, 1e-16))
ax[2].set(xlabel=r"$\xi$", ylabel=r"$|W_{PINN} - W_{exact}|$",
          title="Pointwise error")
ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"max |error|  = {err_soft.max():.3e}  (non-dimensional)")
print(f"             = {err_soft.max() * W_SCALE * 1000:.5f} mm")
print(f"W(0)         = {predict(soft, np.array([0.0]))[0]:+.3e}   (should be exactly 0)")

No labelled deflection anywhere in that loss. The network never saw a
single value of $W$ — only the requirement that its second derivative match
$-3(1-\xi)$, plus two conditions at the support. That is enough to pin the
solution down.

Notice, though: $W(0)$ is *close* to zero, not zero. The penalty pushed it down
but nothing forced it. And we had to pick $\lambda_{BC} = 100$ out of the air.

### Raising $\lambda_{BC}$ makes the solution worse, not better

We picked 100 with no justification. Sweep it over four orders of magnitude and
watch both error measures. (Shorter training here, 3000 epochs, so compare the
columns against each other rather than against the run above.)

In [ ]:
lams = [1.0, 10.0, 100.0, 1000.0, 10000.0]
rows = []
for lam in lams:
    torch.manual_seed(RNG)
    m = make_net()
    train_soft(m, lam_bc=lam, epochs=3000, quiet=True)
    Wp = predict(m, xi_plot)
    rows.append((lam, abs(predict(m, np.array([0.0]))[0]),
                 np.abs(Wp - W_exact(xi_plot)).max()))

print(f"  {'lambda_BC':>10s} {'|W(0)|':>12s} {'max |error|':>13s}")
for lam, bc, err in rows:
    print(f"  {lam:>10.0f} {bc:>12.2e} {err:>13.2e}")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(lams, [r[1] for r in rows], "o-", label="|W(0)| — BC violation")
ax.loglog(lams, [r[2] for r in rows], "s-", label="max solution error")
ax.set(xlabel=r"$\lambda_{BC}$", ylabel="error")
ax.set_title(r"Turning $\lambda_{BC}$ up makes things worse")
ax.legend(); ax.grid(True, which="both", alpha=.3)
plt.tight_layout(); plt.show()

Read that table carefully, because it does **not** say what you might
expect.

Raising $\lambda_{BC}$ does not tighten the boundary condition. The BC violation
sits at around $10^{-6}$ for everything from $\lambda = 1$ to $\lambda = 1000$,
and then gets *worse* at $10^4$. Meanwhile the solution error climbs
monotonically — by $\lambda = 10^4$ it is roughly **three orders of magnitude**
worse than at $\lambda = 1$.

So on this problem the smallest weight wins on both counts, and the instinct many
tutorials encourage — "use a big $\lambda$ to really enforce the boundary
conditions" — is actively destructive. Once $\lambda$ dominates, the optimiser
spends its steps on a term that was already satisfied and stops making progress on
the equation.

Two honest conclusions:

1. **$\lambda_{BC}$ is a hyperparameter you have to sweep.** Our original guess of
   100 was already about 3x worse than $\lambda = 1$, and nothing in the loss
   history would have told us.
2. **Which way it should go is problem-dependent.** Here the BC was easy to satisfy
   so a small weight sufficed. On a problem where the boundary fights the
   interior, the balance flips — and you would have to sweep again.

This is a well-known PINN failure mode: the loss terms have different scales and
different curvature, so their gradients compete. The literature's remedies are
adaptive weighting (learn $\lambda$ during training), gradient-norm balancing, and
sequential training.

Or you can make the question disappear.

## Part 3 — Hard constraints: make the BCs unbreakable

Instead of *penalising* violations, build a network that **cannot** violate them.

Write the trial solution as

$$W_\theta(\xi) = \xi^2 \, N_\theta(\xi)$$

where $N_\theta$ is an ordinary unconstrained MLP. Then, whatever the weights:

- $W_\theta(0) = 0^2 \cdot N(0) = 0$ &nbsp; — the deflection BC, exactly.
- $W_\theta'(\xi) = 2\xi N + \xi^2 N'$, so $W_\theta'(0) = 0$ &nbsp; — the rotation BC, exactly.

The factor $\xi^2$ is not arbitrary: a clamped end needs *two* conditions, and
$\xi^2$ has a double root there. A simple support (only $W = 0$) would take
$\xi^1$; a domain clamped at both ends would take $\xi^2(1-\xi)^2$.

<img src="figs/strong-bc.png" width="620"/>

Both boundary conditions now hold identically, so they leave the loss entirely:

$$\mathcal{L} = \frac{1}{N_c}\sum_i r(\xi_i)^2$$

One term. No $\lambda$ to tune.

In [ ]:
class HardBC(nn.Module):
    """TODO: wrap a base network so that W(xi) = xi**2 * N(xi).
    Convince yourself this forces BOTH W(0)=0 and W'(0)=0.
    """
    ...


def train_hard(model, n_colloc=100, epochs=5000, lr=5e-3, quiet=False):
    """TODO: same as train_soft, but the loss is ONLY physics_loss --
    no boundary term, no lambda.
    """
    ...


torch.manual_seed(RNG)
hard = HardBC(make_net())
h_hard = train_hard(hard, epochs=5000)

In [ ]:
W_hard = predict(hard, xi_plot)
err_hard = np.abs(W_hard - W_exact(xi_plot))

x0 = torch.zeros(1, 1, requires_grad=True)
W0_hard = hard(x0)
slope0 = deriv(W0_hard, x0, 1)

print("                       soft (lambda=100)        hard (xi^2 * N)")
print(f"  |W(0)|          {abs(predict(soft, np.array([0.0]))[0]):>18.3e} "
      f"{abs(float(W0_hard.detach())):>22.3e}")
print(f"  |W'(0)|         {'(not enforced)':>18s} {abs(float(slope0.detach())):>22.3e}")
print(f"  max |error|     {err_soft.max():>18.3e} {err_hard.max():>22.3e}")
print(f"  loss terms      {'2 (+ lambda)':>18s} {'1':>22s}")

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax[0].plot(xi_plot, W_exact(xi_plot) * W_SCALE * 1000, "k--", lw=2.5, label="exact")
ax[0].plot(xi_plot, W_soft * W_SCALE * 1000, lw=1.8, alpha=.8, label="soft BC")
ax[0].plot(xi_plot, W_hard * W_SCALE * 1000, lw=1.8, label="hard BC")
ax[0].set(xlabel=r"$\xi$", ylabel="deflection (mm)", title="Both solve the beam")
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].semilogy(xi_plot, np.maximum(err_soft, 1e-18), label="soft BC")
ax[1].semilogy(xi_plot, np.maximum(err_hard, 1e-18), label="hard BC")
ax[1].set(xlabel=r"$\xi$", ylabel="absolute error", title="Error, note the support")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

The hard-constrained network satisfies both boundary conditions to
machine precision — because they are algebraic identities, not optimisation
targets — and it has one fewer hyperparameter.

**Use hard constraints whenever the geometry lets you.** It is the highest
value-per-line change in this notebook. The limitation is that you need a
distance function you can write down, which is easy on intervals and boxes and
awkward on complicated domains.

## Part 4 — The inverse problem: what PINNs are actually for

Parts 2 and 3 solved a problem a first-year student solves by integrating twice,
and we will be honest about that in Part 5. Here is the case where the PINN earns
its keep.

**The real situation.** You have a beam in the field. You can measure deflection
at a handful of points. You do **not** know its stiffness — the section has
corroded, or the concrete's modulus is uncertain, or it is a soil profile rather
than a beam. You want $EI$.

<img src="figs/inverse-problem-setup.png" width="640"/>

This is hard for a classical solver: it is an optimisation *around* a forward
solve, one solve per iteration. For a PINN it is almost free — make the unknown a
**trainable parameter** and let the same gradient descent that fits the network
also fit the physics.

Write the scaled equation with the unknown exposed. With $\tilde w = w/w_{ref}$
for a fixed reference scale $w_{ref}$,

$$\tilde w''(\xi) = -\kappa\,(1 - \xi), \qquad
\kappa = \frac{PL^3}{EI\,w_{ref}}$$

Recover $\kappa$, and $EI = PL^3 / (\kappa\, w_{ref})$ follows.

We optimise $\log \kappa$ rather than $\kappa$: it keeps $\kappa$ positive
automatically and makes the step size scale-free.

In [ ]:
W_REF = 0.02                     # m, a round number near the measured scale
KAPPA_TRUE = P * L**3 / (EI * W_REF)
print(f"true kappa = {KAPPA_TRUE:.4f}   (this is the answer we must recover)")
print(f"true EI    = {EI:.4e} N m^2")
print(f"true E     = {E / 1e9:.1f} GPa")

# Eight noisy sensor readings, in the scaled variable.
N_SENS, NOISE_MM = 8, 0.15
rng = np.random.default_rng(RNG)
xi_s = np.linspace(0.15, 1.0, N_SENS)          # no sensor at the clamp
w_true_mm = W_exact(xi_s) * W_SCALE * 1000
w_meas_mm = w_true_mm + NOISE_MM * rng.standard_normal(N_SENS)

xi_sens = torch.tensor(xi_s).unsqueeze(1)
w_sens = torch.tensor(w_meas_mm / 1000.0 / W_REF).unsqueeze(1)

print(f"\n{N_SENS} readings with {NOISE_MM} mm noise "
      f"({NOISE_MM / abs(w_true_mm).max() * 100:.1f}% of tip deflection)")
for a, b in zip(w_true_mm, w_meas_mm):
    print(f"    true {a:>8.3f} mm    measured {b:>8.3f} mm")

In [ ]:
class InversePINN(nn.Module):
    """TODO: a hard-BC network (xi^2 * N) that ALSO carries a trainable
    parameter for the unknown stiffness. Store log(kappa) as an
    nn.Parameter and expose kappa = exp(log_kappa) -- why log?
    """
    ...


def inverse_residual(model, xi):
    """TODO: w'' + kappa * (1 - xi), using the model's current kappa."""
    ...


def train_inverse(model, xi_sens, w_sens, epochs=8000, lr=5e-3,
                  n_colloc=100, lam_data=100.0, quiet=False):
    """TODO: minimise  pde_residual + lam_data * data_misfit.
    Track kappa each epoch so you can watch it converge.
    """
    ...


torch.manual_seed(RNG)
inv = InversePINN(make_net(), kappa_init=1.0)
h_inv = train_inverse(inv, xi_sens, w_sens, epochs=8000)

In [ ]:
kappa_hat = float(inv.kappa.detach())
EI_hat = P * L**3 / (kappa_hat * W_REF)
E_hat = EI_hat / I

print(f"  {'':<12s} {'recovered':>14s} {'true':>14s} {'error':>9s}")
print(f"  {'kappa':<12s} {kappa_hat:>14.4f} {KAPPA_TRUE:>14.4f} "
      f"{abs(kappa_hat/KAPPA_TRUE - 1)*100:>8.2f}%")
print(f"  {'EI (N m^2)':<12s} {EI_hat:>14.4e} {EI:>14.4e} "
      f"{abs(EI_hat/EI - 1)*100:>8.2f}%")
print(f"  {'E (GPa)':<12s} {E_hat/1e9:>14.2f} {E/1e9:>14.2f} "
      f"{abs(E_hat/E - 1)*100:>8.2f}%")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].axhline(KAPPA_TRUE, color="k", ls="--", lw=2, label=r"true $\kappa$")
ax[0].plot(h_inv["kappa"], color="crimson", lw=1.8, label=r"$\kappa$ during training")
ax[0].set(xlabel="epoch", ylabel=r"$\kappa$",
          title="The unknown converges alongside the weights")
ax[0].legend(); ax[0].grid(alpha=.3)

w_inv = predict(inv, xi_plot) * W_REF * 1000
ax[1].plot(xi_plot, W_exact(xi_plot) * W_SCALE * 1000, "k--", lw=2.5, label="true beam")
ax[1].plot(xi_plot, w_inv, color="teal", lw=2, label="inverse PINN")
ax[1].scatter(xi_s, w_meas_mm, s=70, color="gold", edgecolor="k",
              zorder=5, label=f"{N_SENS} noisy readings")
ax[1].set(xlabel=r"$\xi$", ylabel="deflection (mm)",
          title="Field recovered, stiffness identified")
ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)

ax[2].semilogy(h_inv["data"], label="data misfit")
ax[2].semilogy(h_inv["pde"], label="PDE residual")
ax[2].set(xlabel="epoch", ylabel="loss", title="Loss components")
ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

Eight noisy readings, no knowledge of the stiffness, and we recovered
$EI$ — and got the full deflection field as a by-product, including at the
support where we never measured.

This is the honest headline for PINNs. Compare the alternatives:

| | Classical inverse solve | PINN |
| --- | --- | --- |
| Structure | outer optimiser wrapping a forward solver | one gradient descent |
| Cost | one full solve per iteration | one backward pass per iteration |
| Sparse, noisy data | needs regularisation bolted on | the PDE *is* the regulariser |
| Extra unknown fields | often intractable | another `nn.Parameter` or another network |

The last row is the real prize: if the stiffness varied along the span,
$EI(x)$, you would replace the scalar parameter with a second small network and
change almost nothing else.

### optional — how much does the noise cost you?

Repeat the identification at several noise levels to see how the estimate
degrades. This is the number a reviewer will ask for.

In [ ]:
noise_levels = [0.0, 0.05, 0.15, 0.4]
out = []
for nz in noise_levels:
    r = np.random.default_rng(RNG)
    meas = W_exact(xi_s) * W_SCALE * 1000 + nz * r.standard_normal(N_SENS)
    ws = torch.tensor(meas / 1000.0 / W_REF).unsqueeze(1)
    torch.manual_seed(RNG)
    m = InversePINN(make_net(), kappa_init=1.0)
    train_inverse(m, xi_sens, ws, epochs=5000, quiet=True)
    k = float(m.kappa.detach())
    out.append((nz, k, P * L**3 / (k * W_REF) / I / 1e9))

print(f"  {'noise (mm)':>11s} {'kappa':>9s} {'E (GPa)':>9s} {'E error':>9s}")
for nz, k, e in out:
    print(f"  {nz:>11.2f} {k:>9.4f} {e:>9.2f} {abs(e*1e9/E - 1)*100:>8.1f}%")

## Part 5 — When PINNs are the wrong tool

Training-course notebooks tend to stop at "it worked". Three caveats that decide
whether you should use this in real work.

### 1. As a forward solver, PINNs lose to FEM. Badly.

Our forward problem has an exact two-element FEM answer. Let's time it.

In [ ]:
# Exact solution by direct integration -- what a solver actually does.
t0 = time.time()
w_direct = W_exact(xi_plot)
t_direct = time.time() - t0

print(f"  analytic / FEM   {t_direct * 1e6:>10.1f} us    error 0 (exact)")
print(f"  PINN (soft BC)   {t_soft:>10.2f} s     error {err_soft.max():.2e}")
print(f"  PINN (hard BC)   {t_hard:>10.2f} s     error {err_hard.max():.2e}")
print(f"\n  PINN is ~{t_hard / max(t_direct, 1e-9):,.0f}x slower and less accurate.")

That ratio is not a bug and it does not go away with tuning. A PINN
solves a global optimisation problem where FEM solves a sparse linear system, and
for a well-posed forward problem on a simple domain the linear system wins every
time.

**So do not sell a PINN as a faster solver.** Reach for one when it offers
something FEM does not:

- **Inverse problems and data assimilation** — Part 4. Unknown coefficients,
  sparse noisy measurements, no clean boundary data.
- **Filling in missing physics** — part of the model is known, part is learned.
- **High-dimensional parametric PDEs** — where meshing is the bottleneck.
- **Awkward geometry or moving boundaries** — no mesh to generate or remesh.

### 2. Higher derivatives get expensive and noisy

We deliberately used $EI\,w'' = M(x)$ rather than $EI\,w'''' = q$. Each extra
derivative means another backward pass through the graph and amplifies the
network's own wiggle. Here is the fourth derivative of our trained network, which
should be exactly zero for a tip-loaded beam ($q = 0$).

In [ ]:
xi_d = torch.linspace(0.02, 0.98, 200, requires_grad=True).unsqueeze(1)
Wd = hard(xi_d)
d2 = deriv(Wd, xi_d, 2).detach().numpy().ravel()
d4 = deriv(hard(xi_d), xi_d, 4).detach().numpy().ravel()
xg = xi_d.detach().numpy().ravel()

fig, ax = plt.subplots(1, 2, figsize=(12.5, 4))
ax[0].plot(xg, Wpp_exact(xg), "k--", lw=2.5, label="exact $W''$")
ax[0].plot(xg, d2, color="teal", lw=1.8, label="AD $W''$")
ax[0].set(xlabel=r"$\xi$", ylabel=r"$W''$", title="2nd derivative: clean")
ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].axhline(0, color="k", ls="--", lw=2.5, label="exact $W'''' = 0$")
ax[1].plot(xg, d4, color="crimson", lw=1.2, label="AD $W''''$")
ax[1].set(xlabel=r"$\xi$", ylabel=r"$W''''$",
          title="4th derivative: noisy, and never trained")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f"  max |W''  - exact| = {np.abs(d2 - Wpp_exact(xg)).max():.3e}")
print(f"  max |W''''|        = {np.abs(d4).max():.3e}   (should be 0)")

The second derivative — the one in the loss — is accurate. The fourth,
which nothing constrained, is orders of magnitude worse. **A PINN is only
accurate in the quantities you put in the loss.** If you need bending moments,
constrain moments; do not differentiate a displacement network twice more and
hope.

For genuinely fourth-order problems the usual fix is a mixed formulation: two
networks, one for $w$ and one for $M$, coupled by $EI w'' = M$ and $M'' = q$.
Two second derivatives instead of one fourth.

### 3. Spectral bias

Networks learn smooth, low-frequency structure first and high-frequency
structure slowly or never. Our beam is a cubic, so this never bit us. A problem
with sharp gradients — a shock, a boundary layer, a liquefaction front — will
train far more reluctantly, and the standard tools are Fourier features, domain
decomposition, and curriculum training on frequency.

### One trained PINN serves exactly one load case

Look back at what we produced: **one deflection field, for one load case.**

Change $P$, and every weight is wrong. Change the load from a tip force to a
distributed pressure and you retrain from scratch — minutes of optimisation for
each new load, when the whole point of a surrogate was instant evaluation.

A design study sweeps hundreds of load cases. A PINN, as built here, cannot serve
one.

That is what operator learning fixes.

## Summary

| | |
| --- | --- |
| **The network is the solution** | Input a coordinate, output a field. Training *is* solving the PDE. |
| **Non-dimensionalise first** | Raw SI put the residual $10^5$ above the solution. This is the most common reason a PINN will not train. |
| **Zero labelled data is enough** | The residual plus boundary conditions pinned down the beam completely. |
| **Soft BCs bring a hyperparameter you cannot win** | No $\lambda_{BC}$ minimised both BC violation and solution error. |
| **Hard constraints are strictly better where available** | $W = \xi^2 N(\xi)$ satisfies the clamped end exactly and deletes $\lambda$. |
| **Inverse problems are the real application** | $EI$ from 8 noisy readings, as one extra `nn.Parameter`. |
| **Accurate only where constrained** | $W''$ was excellent; the unconstrained $W''''$ was garbage. |
| **Not a faster forward solver** | Orders of magnitude slower than FEM here, and less accurate. Be honest about this. |

### Go deeper

- [SciML — PINNs from scratch](https://kks32-courses.github.io/sciml/01-pinns/pinns.html)
- [SciML — soft vs hard constraints](https://kks32-courses.github.io/sciml/01-pinns/poisson.html)
- [SciML — adaptive loss weights](https://kks32-courses.github.io/sciml/01-pinns/adaptive-weights.html)
- [SciML — collocation point strategies](https://kks32-courses.github.io/sciml/01-pinns/collocation.html)
- [SciML — inverse problems](https://kks32-courses.github.io/sciml/01-pinns/inverse-heat.html)
- [SciML — Burgers' equation](https://kks32-courses.github.io/sciml/01-pinns/burgers.html) — a nonlinear, shock-forming case
- [DesignSafe PINN training](https://github.com/DesignSafe-Training/pinn) — heat transfer and Burgers notebooks
- Raissi, Perdikaris & Karniadakis (2019), *Physics-informed neural networks* — the original paper